# Reactivation Loop — Reengajamento de Motoristas via Canal Externo

**Contexto:** Motoristas da 99 ficam inativos após 30 dias sem corridas, impactando disponibilidade de oferta e receita.  
**Solução:** Fluxo de reengajamento via WhatsApp ativado automaticamente a partir dos 15 dias de inatividade, com mensagem personalizada por perfil.

---

## Hipótese
Motoristas contactados via canal externo nos primeiros 15 dias de inatividade reativam mais do que os não contactados.

## Métricas de sucesso
- **Taxa de reativação** ≥ 30% no grupo contactado
- **Janela ideal de intervenção:** dia 7–20, 21–30 ou 31–45?
- **Segmento com melhor resposta** para priorização futura

## 1. Setup e Geração de Dados Simulados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
n = 500

df = pd.DataFrame({
    'motorista_id': range(1, n+1),
    'perfil': np.random.choice(['Novo', 'Recorrente', 'Veterano'], n, p=[0.3, 0.4, 0.3]),
    'dias_inativo': np.random.randint(15, 46, n),
    'contato_externo': np.random.choice([0, 1], n, p=[0.5, 0.5]),
})

df.head()

## 2. Modelagem da Probabilidade de Reativação

In [ ]:
def reativou(row):
    base = 0.15
    if row['contato_externo'] == 1:
        base += 0.20
    if row['perfil'] == 'Veterano':
        base += 0.10
    elif row['perfil'] == 'Recorrente':
        base += 0.05
    if row['dias_inativo'] <= 20:
        base += 0.10
    return np.random.binomial(1, min(base, 1))

df['reativou'] = df.apply(reativou, axis=1)

print(f"Total de motoristas: {len(df)}")
print(f"Taxa geral de reativação: {df['reativou'].mean():.1%}")
print(f"\nDistribuição por grupo:")
print(df.groupby('contato_externo')['reativou'].agg(['count', 'mean']).rename(
    index={0: 'Sem contato', 1: 'Com contato'},
    columns={'count': 'N', 'mean': 'Taxa de Reativação'}
).style.format({'Taxa de Reativação': '{:.1%}'}).to_string())

## 3. Visualizações

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Reactivation Loop — Análise de Reengajamento', fontsize=14, fontweight='bold')

# Gráfico 1 — Contactado vs. Não Contactado
taxa = df.groupby('contato_externo')['reativou'].mean().reset_index()
taxa['contato_externo'] = taxa['contato_externo'].map({0: 'Sem contato', 1: 'Com contato'})
sns.barplot(data=taxa, x='contato_externo', y='reativou', palette=['#cccccc', '#FF6600'], ax=axes[0])
axes[0].set_title('Taxa de Reativação\nContactado vs. Não Contactado')
axes[0].set_ylabel('Taxa de Reativação')
axes[0].set_xlabel('')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))

# Gráfico 2 — Reativação por perfil
taxa_perfil = df.groupby(['perfil', 'contato_externo'])['reativou'].mean().reset_index()
taxa_perfil['contato_externo'] = taxa_perfil['contato_externo'].map({0: 'Sem contato', 1: 'Com contato'})
sns.barplot(data=taxa_perfil, x='perfil', y='reativou', hue='contato_externo',
            palette=['#cccccc', '#FF6600'], ax=axes[1])
axes[1].set_title('Reativação por Perfil\nde Motorista')
axes[1].set_ylabel('Taxa de Reativação')
axes[1].set_xlabel('')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))

# Gráfico 3 — Janela ideal de intervenção
df['janela'] = pd.cut(df['dias_inativo'], bins=[14, 20, 30, 45],
                      labels=['7-20 dias', '21-30 dias', '31-45 dias'])
taxa_janela = df[df['contato_externo'] == 1].groupby('janela', observed=True)['reativou'].mean().reset_index()
sns.barplot(data=taxa_janela, x='janela', y='reativou', color='#FF6600', ax=axes[2])
axes[2].set_title('Janela Ideal de Intervenção\n(Grupo Contactado)')
axes[2].set_ylabel('Taxa de Reativação')
axes[2].set_xlabel('Dias de Inatividade')
axes[2].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))

plt.tight_layout()
plt.savefig('reactivation_loop.png', dpi=150)
plt.show()

## 4. Análise Detalhada por Segmento

In [ ]:
resumo = df[df['contato_externo'] == 1].groupby('perfil')['reativou'].agg(
    N='count',
    Reativados='sum',
    Taxa='mean'
).sort_values('Taxa', ascending=False)

resumo['Taxa'] = resumo['Taxa'].map('{:.1%}'.format)
print("Grupo Contactado — Reativação por Perfil:")
print(resumo.to_string())

In [ ]:
resumo_janela = df[df['contato_externo'] == 1].groupby('janela', observed=True)['reativou'].agg(
    N='count',
    Reativados='sum',
    Taxa='mean'
)
resumo_janela['Taxa'] = resumo_janela['Taxa'].map('{:.1%}'.format)
print("Grupo Contactado — Reativação por Janela de Intervenção:")
print(resumo_janela.to_string())

## 5. Conclusões e Recomendações

### O que os dados mostram

| Achado | Resultado |
|--------|----------|
| Contactados reativam mais | Diferença de ~20pp vs. grupo controle |
| Melhor janela | 7–20 dias de inatividade |
| Melhor segmento | Veteranos (+12 meses na plataforma) |

### Recomendação
Priorizar contato externo via WhatsApp nos **primeiros 15 dias de inatividade**, com mensagem personalizada por perfil. Começar pelo segmento **Veterano**, que tem maior taxa de retorno e maior valor histórico para a plataforma.

### Próximos passos
1. Validar hipótese com dados reais de uma coorte pequena
2. Testar duas versões de mensagem (A/B) para otimizar conversão
3. Definir SLA de resposta do motorista para considerar reativação